# ST-TBA-GAT MARL Training on Kaggle GPU
**Decentralized Industrial Hazard Evacuation**

This notebook trains the Spatio-Temporal Topological Bottleneck-Aware Graph Attention Network (ST-TBA-GAT) using MAPPO across 36 edge router signboards.

### Hardware Requirement
Enable GPU in Kaggle settings: **Notebook Settings -> Accelerator -> GPU T4 x2 (or P100)**.

In [ ]:
# 1. Check GPU Availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
else:
    print("WARNING: CUDA is not available. Please enable GPU in Notebook Settings.")

In [ ]:
# 2. Clone Repository & Checkout Training Branch
import os, shutil
repo_dir = "/kaggle/working/decentralized_industrial_hazard_evacuation"
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

!git clone -b marl-v2-training https://github.com/kitretsu2809/decentralized_industrial_hazard_evacuation.git
%cd /kaggle/working/decentralized_industrial_hazard_evacuation

In [ ]:
# 3. Install Dependencies
!pip install -q gymnasium pettingzoo onnx

In [ ]:
# 4. Verify Environment Unit Tests (34/34)
!PYTHONPATH=. python -m unittest discover tests -v

In [ ]:
# 5. Train ST-TBA-GAT MAPPO on GPU
# - 4-stage pedagogical curriculum (1 to 100 episodes)
# - Topological cycle & exit reachability penalties
# - Zero-in-transit episode termination
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

!PYTHONPATH=. python simulator/training/train_mappo.py \
    --device cuda \
    --episodes 100 \
    --save-dir /kaggle/working/checkpoints \
    --min-evacuees 30 \
    --max-evacuees 400 \
    --curriculum \
    --eval

In [ ]:
# 6. Copy ONNX and verify outputs
if os.path.exists("data/models/policy.onnx"):
    shutil.copy("data/models/policy.onnx", "/kaggle/working/checkpoints/policy.onnx")

print("\n--- Output Files in /kaggle/working/checkpoints ---")
for f in os.listdir("/kaggle/working/checkpoints"):
    p = os.path.join("/kaggle/working/checkpoints", f)
    print(f"  {f} ({os.path.getsize(p)/1024:.1f} KB)")

In [ ]:
# 7. Generate Direct Download Links
from IPython.display import FileLink, display

if os.path.exists("/kaggle/working/checkpoints/best_policy.pt"):
    display(FileLink("/kaggle/working/checkpoints/best_policy.pt"))
if os.path.exists("/kaggle/working/checkpoints/policy.onnx"):
    display(FileLink("/kaggle/working/checkpoints/policy.onnx"))
print("\nClick the links above or use the Kaggle right-hand panel (Data > Output) to download!")

### How to Use the Downloaded Model Locally:
1. Download `best_policy.pt` from Kaggle.
2. Copy the file into your local project:
   ```bash
   cp ~/Downloads/best_policy.pt checkpoints/directional/best_policy.pt
   cp ~/Downloads/best_policy.pt data/models/st_tba_gat_latest.pt
   ```
3. Open the simulator at `http://127.0.0.1:8080/` (or toggle mode to MARL). The simulation immediately loads the newly trained neural policy!